# Lecture 10 — Recursion
## Worked examples

**Programming · Human-Centered Artificial Intelligence · Università degli Studi di Milano**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramonpeter/programming-hcai-2026/blob/main/lectures/10-recursion-demo.ipynb)

The concepts are on the slides. This notebook solves the two problems from the lecture,
including the wrong turns.

To watch the call stack grow and shrink, paste any of these functions into
[Python Tutor](https://pythontutor.com) and step through it.

---
## A · Fibonacci, and why `fib(35)` is slow

### Step 1 · Straight from the definition

F(0) = 0, F(1) = 1, and F(n) = F(n − 1) + F(n − 2). A first attempt, with the base case
that comes to mind first.

In [ ]:
def fib(n):
    """Return the n-th Fibonacci number."""
    if n == 1:
        return 1
    return fib(n - 1) + fib(n - 2)

fib(5)

Question 1 of the three: is there a base case? Yes, but question 2 fails. `fib(2)` asks
for `fib(1)`, which is fine, and for `fib(0)`, which is not a base case. So `fib(0)` asks
for `fib(-1)` and `fib(-2)`, and the numbers run away from 1 forever.

The step goes down by 1 *and* by 2, so two neighbouring values must both be base cases.
F(0) = 0 and F(1) = 1, so for both of them the answer is `n` itself.

In [ ]:
def fib(n):
    """Return the n-th Fibonacci number."""
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

[fib(n) for n in range(10)]

Correct. Now try bigger numbers, and time them.

In [ ]:
import time

for n in [20, 25, 30]:
    start = time.time()
    value = fib(n)
    print(n, value, f"{time.time() - start:.2f} s")

Each step of 5 makes it about ten times slower. `fib(35)` takes a couple of seconds,
and `fib(50)` would take most of an hour. The function is short and correct, so where does the time go?

**Try it:** run `fib(35)` once and wait for it.

### Step 2 · Count the calls

A counter that goes up by one in every call.

In [ ]:
calls = 0

def fib(n):
    """Return the n-th Fibonacci number."""
    calls += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

fib(10)

Lecture 8. `calls += 1` assigns to `calls`, so inside the function `calls` is a local
name, and it has no value yet. We do not want `global`. A dictionary works, because
assigning to `counts[n]` changes a box inside the dictionary and never moves the name
`counts`.
And with one entry per `n`, it shows *which* calls happen, not only how many.

In [ ]:
counts = {}

def fib(n):
    """Return the n-th Fibonacci number."""
    counts[n] = counts.get(n, 0) + 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

fib(10)
print(dict(sorted(counts.items())))
print("calls:", sum(counts.values()))

177 calls to compute one number from 11 different ones. `fib(1)` alone is computed 55
times. Every call splits into two, and both halves recompute the same smaller values.
The counts are Fibonacci numbers themselves, so the work grows like the answer does.

In [ ]:
for n in [10, 20, 30]:
    counts = {}
    fib(n)
    print(n, sum(counts.values()))

Almost 2.7 million calls for `fib(30)`, to combine 31 different values. The recursion is
not slow. It is doing the same work over and over.

### Step 3 · A fast version

To compute F(n) we only ever need the previous two numbers. A loop that carries two
values along does that once each.

In [ ]:
def fib_loop(n):
    """Return the n-th Fibonacci number, with a loop."""
    a, b = 0, 1           # F(0) and F(1)
    for _ in range(n):
        a, b = b, a + b
    return a

print([fib_loop(n) for n in range(10)])
print(fib_loop(30), fib_loop(100))

The same answers, and `fib_loop(100)` is instant. The tuple assignment from lecture 7
moves both numbers one step along at once.

The loop is faster, and the recursive version still reads like the definition. Block B
shows how to keep the recursive version and make it fast as well.

---
## B · Searching a sorted list

### Step 1 · The slow way

A million even numbers, sorted.

In [ ]:
xs = list(range(0, 2_000_000, 2))
print(len(xs), xs[:5], xs[-1])
print(1234 in xs, 1235 in xs)
print(xs.index(1234))

`in` and `index` work on any list, so they cannot use the order. They look at the items
one by one. For a number near the end, or one that is not there, that is a million
comparisons.

The guessing game from the slide uses the order. Look at the middle. If the target is
bigger, it can only be in the right half, so the left half is gone in one comparison.

### Step 2 · Halves, the first idea

Look at the middle, then search the half that is left. The half is a slice.

In [ ]:
def contains(xs, target):
    """Return True if target is in the sorted list xs."""
    if xs == []:
        return False
    mid = len(xs) // 2
    if xs[mid] == target:
        return True
    if xs[mid] < target:
        return contains(xs[mid + 1:], target)
    return contains(xs[:mid], target)

print(contains(xs, 1234), contains(xs, 1235))

Correct, with two problems. It can only say yes or no. A slice is a new list numbered from
0, so the position of the target in `xs` is lost on the way down. And each slice copies
half of the list, so the first call alone copies half a million numbers, which is more
work than looking at all of them.

Better: never copy. Keep the whole list, and two positions `lo` and `hi` that mark the
part still to search, from `lo` up to but not including `hi`, like `range`.

### Step 3 · Positions instead of slices

In [ ]:
def search(xs, target, lo, hi):
    """Return the position of target in xs[lo:hi], or -1. xs is sorted."""
    if lo >= hi:
        return -1
    mid = (lo + hi) // 2
    if xs[mid] == target:
        return mid
    if xs[mid] < target:
        return search(xs, target, mid, hi)
    return search(xs, target, lo, mid)

print(search(xs, 1234, 0, len(xs)))

617, and `xs[617]` is indeed 1234. Now a number that is not there.

In [ ]:
print(search(xs, 1235, 0, len(xs)))

It works for every number that is in the list, and never stops for one that is not. Trace
the end on paper, with a small list.

In [ ]:
small = [2, 3, 5, 7, 11, 13, 17, 19]
trace = {"calls": 0}

def search(xs, target, lo, hi):
    """Return the position of target in xs[lo:hi], or -1. xs is sorted."""
    print("searching", lo, hi)
    trace["calls"] += 1
    if trace["calls"] == 6:
        return "stopped after six calls"
    if lo >= hi:
        return -1
    mid = (lo + hi) // 2
    if xs[mid] == target:
        return mid
    if xs[mid] < target:
        return search(xs, target, mid, hi)
    return search(xs, target, lo, mid)

search(small, 4, 0, len(small))

The two lines with `trace` stop the search after six calls, so that it prints six lines
instead of a thousand. They are only for looking.

`searching 1 2`, again and again. With `lo = 1` and `hi = 2` the middle is 1, `small[1]` is
3, which is less than 4, so the next call gets `lo = mid = 1` again. The range never
shrinks, and question 2 fails. `mid` has already been checked, so the right half starts
at `mid + 1`.

In [ ]:
def search(xs, target, lo, hi):
    """Return the position of target in xs[lo:hi], or -1. xs is sorted."""
    if lo >= hi:
        return -1
    mid = (lo + hi) // 2
    if xs[mid] == target:
        return mid
    if xs[mid] < target:
        return search(xs, target, mid + 1, hi)
    return search(xs, target, lo, mid)


def find(xs, target):
    """Return the position of target in the sorted list xs, or -1."""
    return search(xs, target, 0, len(xs))


print(find(small, 13), find(small, 4), find(small, 2), find(small, 19), find(small, 20))
print(find(xs, 1234), find(xs, 1235), find(xs, 1_999_998))

`find` is there so that nobody has to remember to pass `0, len(xs)`. The first, the last,
a missing number in the middle and one past the end all work.

### Step 4 · How many steps?

Count the calls, with a dictionary as in part A.

In [ ]:
steps = {"calls": 0}

def search(xs, target, lo, hi):
    """Return the position of target in xs[lo:hi], or -1. xs is sorted."""
    steps["calls"] += 1
    if lo >= hi:
        return -1
    mid = (lo + hi) // 2
    if xs[mid] == target:
        return mid
    if xs[mid] < target:
        return search(xs, target, mid + 1, hi)
    return search(xs, target, lo, mid)

for target in [1234, 1235, 1_999_998, -1]:
    steps["calls"] = 0
    print(target, find(xs, target), steps["calls"], "calls")

Never more than 21 calls for a million items, where `in` needs up to a million comparisons.
The recursion is at most 21 deep, far from the limit of 1000.

### Step 5 · The same with a loop

The recursive call only changes `lo` or `hi`, and nothing happens after it. So a `while`
loop can do the same by changing `lo` and `hi` itself.

In [ ]:
def find_loop(xs, target):
    """Return the position of target in the sorted list xs, or -1."""
    lo, hi = 0, len(xs)
    while lo < hi:
        mid = (lo + hi) // 2
        if xs[mid] == target:
            return mid
        if xs[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return -1

print(find_loop(small, 13), find_loop(small, 4), find_loop(xs, 1234))

Put the two side by side. The base case `lo >= hi` became the loop condition `lo < hi`,
turned around. Each recursive call became one assignment. Neither is better here: the
recursive one is closer to the idea, the loop one has no depth limit at all.

Check that they agree, on every number from −2 to 21 and on the list method.

In [ ]:
for t in range(-2, 22):
    expected = small.index(t) if t in small else -1
    if find(small, t) != expected or find_loop(small, t) != expected:
        print("disagree on", t)
print("checked")

Only `checked`. Python's own version of this lives in the module `bisect`, lecture 9 style.

### Step 6 · The Towers of Hanoi, if time allows

Move a tower of `n` discs from peg A to peg C, one disc at a time, never putting a larger
disc on a smaller one. Peg B helps.

The three questions answer it. One disc: just move it. More discs: move the top `n − 1`
out of the way onto the spare peg, move the bottom disc, then move the `n − 1` back on top
of it. Trust the smaller call.

In [ ]:
def hanoi(n, source, target, spare):
    """Print the moves that take n discs from source to target."""
    if n == 1:
        print(f"move disc 1 from {source} to {target}")
        return
    hanoi(n - 1, source, spare, target)
    print(f"move disc {n} from {source} to {target}")
    hanoi(n - 1, spare, target, source)

hanoi(3, "A", "C", "B")

Seven moves for three discs. Nobody can follow all of the calls in their head, and the
function is still obviously right, which is what the leap of faith is for. Each extra
disc doubles the moves and adds one: 2ⁿ − 1.

**Try it:** change `print` into a counter and check 2ⁿ − 1 for n from 1 to 10.

---
## Where we are going

Next lecture: **classes and objects**. Data and the functions that work on it, kept
together.